<a href="https://colab.research.google.com/github/prksh830/Healthcare/blob/main/HGWGO_AAME_CVD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

In [3]:
cvd = pd.read_csv("CVD_cleaned.csv")
cvd = cvd.drop_duplicates()

In [4]:
y = cvd["Heart_Disease"]

In [5]:
X = cvd.drop("Heart_Disease", axis=1)

In [6]:
import pandas as pd

cvd = pd.read_csv("CVD_cleaned.csv")
cvd = cvd.drop_duplicates().reset_index(drop=True)

print(cvd.shape)
print(cvd.dtypes)

(308774, 19)
General_Health                   object
Checkup                          object
Exercise                         object
Skin_Cancer                      object
Other_Cancer                     object
Depression                       object
Diabetes                         object
Arthritis                        object
Sex                              object
Age_Category                     object
Height_(cm)                       int64
Weight_(kg)                     float64
BMI                             float64
Smoking_History                  object
Alcohol_Consumption               int64
Fruit_Consumption                 int64
Green_Vegetables_Consumption      int64
FriedPotato_Consumption           int64
Heart_Disease                    object
dtype: object


In [7]:
print(cvd["Heart_Disease"].value_counts())

Heart_Disease
No     283803
Yes     24971
Name: count, dtype: int64


In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [9]:
cvd = pd.read_csv("CVD_cleaned.csv")

cvd = cvd.drop_duplicates().reset_index(drop=True)

print(cvd.shape)

(308774, 19)


In [10]:
cvd["Heart_Disease"] = cvd["Heart_Disease"].map({
    "No":0,
    "Yes":1
})

In [12]:
cat_cols = cvd.select_dtypes(include="object").columns.tolist()

print(cat_cols)
print("Number of categorical columns:", len(cat_cols))

['General_Health', 'Checkup', 'Exercise', 'Skin_Cancer', 'Other_Cancer', 'Depression', 'Diabetes', 'Arthritis', 'Sex', 'Age_Category', 'Smoking_History']
Number of categorical columns: 11


In [13]:
encoders = {}

for col in cat_cols:

    le = LabelEncoder()

    cvd[col] = le.fit_transform(cvd[col])

    encoders[col] = le

In [14]:
X = cvd.drop("Heart_Disease", axis=1)

y = cvd["Heart_Disease"]

print(X.shape)
print(y.shape)

(308774, 18)
(308774,)


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(247019, 18)
(61755, 18)


In [16]:
!pip install -q imbalanced-learn

In [17]:
from imblearn.over_sampling import SMOTENC

In [18]:
cat_indices = [
    X.columns.get_loc(col)
    for col in cat_cols
]

In [21]:
X = cvd.drop("Heart_Disease", axis=1)
y = cvd["Heart_Disease"]

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [23]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

In [24]:
print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(247019, 18)
(61755, 18)
Heart_Disease
0    227042
1     19977
Name: count, dtype: int64
Heart_Disease
0    56761
1     4994
Name: count, dtype: int64


In [25]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

In [26]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)

pred = rf.predict(X_test)
prob = rf.predict_proba(X_test)[:,1]

print("Accuracy :", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall   :", recall_score(y_test,pred))
print("F1       :", f1_score(y_test,pred))
print("AUROC    :", roc_auc_score(y_test,prob))
print("MCC      :", matthews_corrcoef(y_test,pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,pred))

Accuracy : 0.9189053517933771
Precision: 0.472
Recall   : 0.023628354024829795
F1       : 0.04500381388253242
AUROC    : 0.8222890547884396
MCC      : 0.09146628490081928

Confusion Matrix
[[56629   132]
 [ 4876   118]]


In [27]:
import numpy as np

thresholds = [0.10,0.15,0.20,0.25,0.30]

for t in thresholds:

    pred_t = (prob >= t).astype(int)

    print("\nThreshold:", t)

    print("Precision:",
          precision_score(y_test,pred_t))

    print("Recall:",
          recall_score(y_test,pred_t))

    print("F1:",
          f1_score(y_test,pred_t))

    print("MCC:",
          matthews_corrcoef(y_test,pred_t))


Threshold: 0.1
Precision: 0.2103832870973221
Recall: 0.7440929114937925
F1: 0.32802224478086245
MCC: 0.3006743543194793

Threshold: 0.15
Precision: 0.2555450073358074
Recall: 0.592911493792551
F1: 0.35715578071286413
MCC: 0.3079149849833999

Threshold: 0.2
Precision: 0.29249119002439683
Recall: 0.43211854225070084
F1: 0.348852247009376
MCC: 0.285922221354188

Threshold: 0.25
Precision: 0.32734746862829944
Recall: 0.30296355626752103
F1: 0.3146838602329451
MCC: 0.2571432557806713

Threshold: 0.3
Precision: 0.3669889008234873
Recall: 0.2052462955546656
F1: 0.26325927828432
MCC: 0.22841355250281453


In [28]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [29]:
from xgboost import XGBClassifier

In [30]:
from lightgbm import LGBMClassifier

In [32]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00


In [33]:
from catboost import CatBoostClassifier

In [34]:
from catboost import CatBoostClassifier

cat = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=100
)

In [35]:
cat.fit(
    X_train,
    y_train
)

0:	total: 147ms	remaining: 43.9s
100:	total: 9.85s	remaining: 19.4s
200:	total: 16.3s	remaining: 8.01s
299:	total: 21.7s	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=300, learning_rate=0.1, loss_function='Logloss', verbose=100)

In [36]:
pred = cat.predict(X_test)

prob = cat.predict_proba(X_test)[:,1]

from sklearn.metrics import *

print("Accuracy :", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall   :", recall_score(y_test,pred))
print("F1       :", f1_score(y_test,pred))
print("AUROC    :", roc_auc_score(y_test,prob))
print("MCC      :", matthews_corrcoef(y_test,pred))

Accuracy : 0.9198445470002429
Precision: 0.5632183908045977
Recall   : 0.03924709651581898
F1       : 0.07338075627105953
AUROC    : 0.8406125475339175
MCC      : 0.13318833905435423


In [37]:
from sklearn.metrics import *

thresholds = [0.05,0.10,0.15,0.20,0.25,0.30]

for t in thresholds:

    pred_t = (prob >= t).astype(int)

    print("\nThreshold:", t)

    print("Precision:",
          precision_score(y_test,pred_t))

    print("Recall:",
          recall_score(y_test,pred_t))

    print("F1:",
          f1_score(y_test,pred_t))

    print("MCC:",
          matthews_corrcoef(y_test,pred_t))


Threshold: 0.05
Precision: 0.16846717534184483
Recall: 0.8930716860232278
F1: 0.2834625651455447
MCC: 0.278331565931834

Threshold: 0.1
Precision: 0.22118722517935663
Recall: 0.7655186223468162
F1: 0.34320854654816413
MCC: 0.3208661212815617

Threshold: 0.15
Precision: 0.26823387582881253
Recall: 0.6237484981978374
F1: 0.3751430119828988
MCC: 0.33073857247774263

Threshold: 0.2
Precision: 0.30797290255341325
Recall: 0.47336804164998
F1: 0.37316495659037097
MCC: 0.3138358901329007

Threshold: 0.25
Precision: 0.34780035509962515
Recall: 0.35302362835402484
F1: 0.3503925270793998
MCC: 0.2927840294920592

Threshold: 0.3
Precision: 0.39408866995073893
Recall: 0.2563075690828995
F1: 0.3106042222761466
MCC: 0.2706932475878939


In [38]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [39]:
pred = xgb.predict(X_test)

prob = xgb.predict_proba(X_test)[:,1]

from sklearn.metrics import *

print("Accuracy :", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall   :", recall_score(y_test,pred))
print("F1       :", f1_score(y_test,pred))
print("AUROC    :", roc_auc_score(y_test,prob))
print("MCC      :", matthews_corrcoef(y_test,pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,pred))

Accuracy : 0.9194559145008502
Precision: 0.521551724137931
Recall   : 0.048458149779735685
F1       : 0.0886771711249542
AUROC    : 0.8378130499433308
MCC      : 0.14064054441740248

Confusion Matrix
[[56539   222]
 [ 4752   242]]


In [40]:
thresholds = [0.05,0.10,0.15,0.20,0.25,0.30]

for t in thresholds:

    pred_t = (prob >= t).astype(int)

    print("\nThreshold:", t)

    print("Precision:",
          precision_score(y_test,pred_t))

    print("Recall:",
          recall_score(y_test,pred_t))

    print("F1:",
          f1_score(y_test,pred_t))

    print("MCC:",
          matthews_corrcoef(y_test,pred_t))


Threshold: 0.05
Precision: 0.17151050143377508
Recall: 0.8862635162194633
F1: 0.2874025974025974
MCC: 0.28169031709535586

Threshold: 0.1
Precision: 0.2213691243699449
Recall: 0.7563075690828995
F1: 0.34249183895538626
MCC: 0.3184183004256647

Threshold: 0.15
Precision: 0.263891293058681
Recall: 0.6105326391670004
F1: 0.3685037466763355
MCC: 0.32206188382311696

Threshold: 0.2
Precision: 0.30371713508612874
Recall: 0.4695634761714057
F1: 0.3688556822650413
MCC: 0.30898479835914766

Threshold: 0.25
Precision: 0.33714721586575136
Recall: 0.3540248297957549
F1: 0.345379957022856
MCC: 0.286352620222031

Threshold: 0.3
Precision: 0.38439635535307515
Recall: 0.27032438926712055
F1: 0.31742299553256526
MCC: 0.2733868174804315


In [41]:
from sklearn.ensemble import ExtraTreesClassifier

In [42]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

et.fit(X_train, y_train)

pred = et.predict(X_test)
prob = et.predict_proba(X_test)[:,1]

In [43]:
print("Extra Trees Training Completed")

Extra Trees Training Completed


In [44]:
from sklearn.metrics import *

pred = et.predict(X_test)

prob = et.predict_proba(X_test)[:,1]

print("Accuracy :", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall   :", recall_score(y_test,pred))
print("F1       :", f1_score(y_test,pred))
print("AUROC    :", roc_auc_score(y_test,prob))
print("MCC      :", matthews_corrcoef(y_test,pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,pred))

Accuracy : 0.9172374706501498
Precision: 0.40134907251264756
Recall   : 0.04765718862635162
F1       : 0.08519778056201897
AUROC    : 0.8101040869910332
MCC      : 0.11574754917594227

Confusion Matrix
[[56406   355]
 [ 4756   238]]


In [45]:
thresholds = [0.05,0.10,0.15,0.20,0.25,0.30]

for t in thresholds:

    pred_t = (prob >= t).astype(int)

    print("\nThreshold:", t)

    print("Precision:",
          precision_score(y_test,pred_t))

    print("Recall:",
          recall_score(y_test,pred_t))

    print("F1:",
          f1_score(y_test,pred_t))

    print("MCC:",
          matthews_corrcoef(y_test,pred_t))


Threshold: 0.05
Precision: 0.1623540966267518
Recall: 0.8606327593111734
F1: 0.27317507229796295
MCC: 0.25889974508677177

Threshold: 0.1
Precision: 0.20457006097213518
Recall: 0.7188626351621946
F1: 0.31850241760191633
MCC: 0.28588116322713203

Threshold: 0.15
Precision: 0.23679447095606385
Recall: 0.5762915498598318
F1: 0.33566596687660366
MCC: 0.2831113222931175

Threshold: 0.2
Precision: 0.26788486802818584
Recall: 0.4491389667601121
F1: 0.33560260342634846
MCC: 0.2716732437984775

Threshold: 0.25
Precision: 0.2944859975647939
Recall: 0.3390068081698038
F1: 0.31518197896304573
MCC: 0.2510381142610263

Threshold: 0.3
Precision: 0.32380216383307575
Recall: 0.25170204245094113
F1: 0.2832356917530419
MCC: 0.23078166178988013
